# 🤖 Lab 2.4 — `generate_evals_df` — Synthetic Dataset Generation

## Learning Objectives
In this notebook, you will learn:
1. **Synthetic Generation** — Use `generate_evals_df` to auto-create representative Q&A pairs from your docs
2. **Coverage Strategy** — Bias the generator with `agent_description` and `question_guidelines` for edge-case coverage
3. **Schema Compatibility** — Confirm the output is already in MLflow's eval schema — no remapping needed
4. **Diversity Audit** — Inspect topic spread across source documents to validate coverage
5. **End-to-End Eval** — Pipe the synthetic set straight into `mlflow.genai.evaluate()` with the Correctness judge

## Prerequisites
- Lab 1.3 completed
- `databricks-agents` SDK installed (handled in Step 1)
- Foundation Model API access


---
## Step 1 — Install Packages

`generate_evals_df` lives in `databricks-agents`. We also pull `mlflow[databricks]>=3.1` and `databricks-openai` so the same notebook can run an end-to-end `evaluate()` after generation.


In [ ]:
# ============================================================================
# 📦 STEP 1 - INSTALL PACKAGES
# ============================================================================

%pip install --quiet "mlflow[databricks]>=3.1" databricks-agents databricks-openai

dbutils.library.restartPython()


---
## Step 2 — Configure the Tutorial Namespace


In [ ]:
# ============================================================================
# 🗂️ STEP 2 - CONFIGURE THE TUTORIAL NAMESPACE
# ============================================================================

import mlflow

CATALOG = "genai_eval_tutorial"
SCHEMA  = "module_01"

USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)
EXPERIMENT_PATH = f"/Users/{USER_EMAIL}/genai-eval-tutorial"
mlflow.set_experiment(EXPERIMENT_PATH)

print(f"Experiment: {EXPERIMENT_PATH}")
print(f"Namespace:  {CATALOG}.{SCHEMA}")


---
## Step 3 — Prepare Documentation Chunks

`generate_evals_df` accepts a Spark/pandas DataFrame with a `content` column (and optionally a `doc_uri`). For real workloads you would load from a Delta table or UC Volume; we hand-craft 6 Delta Lake chunks for reproducibility.


In [ ]:
# ============================================================================
# 📄 STEP 3 - PREPARE DOCUMENTATION CHUNKS
# ============================================================================

import pandas as pd

docs = [
    {
        "doc_uri": "delta_lake/overview.md",
        "content": (
            "Delta Lake is an open-source storage layer that provides ACID transactions, "
            "scalable metadata handling, and unified streaming and batch data processing "
            "on top of cloud object stores."
        ),
    },
    {
        "doc_uri": "delta_lake/vacuum.md",
        "content": (
            "VACUUM removes data files no longer referenced by a Delta table that are older "
            "than the retention threshold (default 7 days). Reducing the threshold below "
            "the default can break time travel and concurrent readers."
        ),
    },
    {
        "doc_uri": "delta_lake/zorder.md",
        "content": (
            "Z-ordering is a technique to co-locate related information in the same set of files. "
            "It improves data skipping and therefore query performance for selective filters."
        ),
    },
    {
        "doc_uri": "delta_lake/time_travel.md",
        "content": (
            "Delta Lake supports time travel — querying older snapshots of a table by version "
            "number or timestamp using AS OF clauses. Useful for audits, rollbacks, and reproducible ML."
        ),
    },
    {
        "doc_uri": "delta_lake/schema_evolution.md",
        "content": (
            "Delta Lake enforces schema by default but supports schema evolution when writing "
            "with mergeSchema=true or via ALTER TABLE. Type changes require explicit migration."
        ),
    },
    {
        "doc_uri": "delta_lake/optimize.md",
        "content": (
            "OPTIMIZE compacts small files into larger ones to improve read performance. "
            "Combine with Z-ORDER BY for filter columns to maximise data skipping benefits."
        ),
    },
]

docs_df = spark.createDataFrame(pd.DataFrame(docs))
display(docs_df)


---
## Step 4 — Generate 20 Synthetic Evals

`generate_evals_df` uses Databricks-hosted LLMs to:
- Read each chunk
- Produce diverse Q&A pairs covering main topics, paraphrases, and edge cases
- Emit the result in MLflow's evaluation schema (`inputs`, `expectations`, `expected_retrieved_context`) — ready for `evaluate()` with no further mapping

`agent_description` and `question_guidelines` strongly bias the generator — make them specific.


In [ ]:
# ============================================================================
# 🤖 STEP 4 - GENERATE 20 SYNTHETIC EVALS
# ============================================================================

from databricks.agents.evals import generate_evals_df

agent_description = (
    "A Q&A assistant for Databricks and Delta Lake. Users ask short, direct questions and "
    "expect concise, factual answers grounded in the documentation provided."
)

question_guidelines = """
- Each question should be answerable from the documentation chunks alone.
- Cover happy-path topics AND edge cases (e.g. what breaks when retention is reduced below 7 days).
- Mix difficulty: half basic definitions, half operational / behavioural questions.
- Phrase questions naturally — as a Databricks engineer would type them.
"""

synthetic_eval_df = generate_evals_df(
    docs=docs_df,
    num_evals=20,
    agent_description=agent_description,
    question_guidelines=question_guidelines,
)

print(f"Generated {synthetic_eval_df.count()} synthetic evals.")


---
## Step 5 — Review Diversity

Eyeball the questions, expected facts, and expected retrieved chunks. Quick checks:
- Are questions paraphrased differently or just slightly reworded?
- Do `expected_facts` actually appear in the cited doc?
- Are edge cases (e.g. retention thresholds, schema evolution failures) represented?


In [ ]:
# ============================================================================
# 🔍 STEP 5 - REVIEW DIVERSITY
# ============================================================================

display(synthetic_eval_df)


In [ ]:
# ============================================================================
# 🔍 TOPIC SPREAD — GROUP BY SOURCE DOC TO CONFIRM COVERAGE
# ============================================================================

from pyspark.sql import functions as F

if "expected_retrieved_context" in synthetic_eval_df.columns:
    coverage = (
        synthetic_eval_df
        .withColumn("doc", F.explode("expected_retrieved_context.doc_uri"))
        .groupBy("doc")
        .count()
        .orderBy(F.desc("count"))
    )
    display(coverage)
else:
    print("Schema does not include expected_retrieved_context in this MLflow version.")


---
## Step 6 — (Optional) Persist to UC for Reuse

Synthetic generation costs tokens — save the result so you don't regenerate it for every notebook run.


In [ ]:
# ============================================================================
# 💾 STEP 6 - (OPTIONAL) PERSIST TO UC FOR REUSE
# ============================================================================

SYNTHETIC_FQN = f"{CATALOG}.{SCHEMA}.tutorial_eval_synthetic_v1"
synthetic_eval_df.write.mode("overwrite").saveAsTable(SYNTHETIC_FQN)
print(f"Saved synthetic eval set: {SYNTHETIC_FQN}")


---
## Step 7 — Use Directly in `mlflow.genai.evaluate`

The DataFrame is already in the right schema — no mapping, no renames. We wire up the same `my_agent` shape from Lab 2.3 and run the **Correctness** judge over the synthetic set.


In [ ]:
# ============================================================================
# 🧮 STEP 7 - USE DIRECTLY IN `MLFLOW.GENAI.EVALUATE`
# ============================================================================

from databricks.sdk import WorkspaceClient
from mlflow.genai.scorers import Correctness

client = WorkspaceClient().serving_endpoints.get_open_ai_client()

def my_agent(question: str) -> str:
    resp = client.chat.completions.create(
        model="databricks-claude-sonnet-4",
        messages=[
            {"role": "system", "content": "You are a concise Databricks expert. Answer in 2 sentences."},
            {"role": "user",   "content": question},
        ],
    )
    return resp.choices[0].message.content

results = mlflow.genai.evaluate(
    data=synthetic_eval_df,
    predict_fn=lambda question: my_agent(question),
    scorers=[Correctness()],
)

display(results.tables["eval_results"])


---
## Lab Complete

| Check | Status |
| --- | --- |
| 6 doc chunks loaded into a DataFrame | ✅ |
| `generate_evals_df` produced 20 Q&A pairs | ✅ |
| Coverage spread across multiple source docs | ✅ |
| Synthetic set persisted as `tutorial_eval_synthetic_v1` | ✅ |
| `evaluate()` ran end-to-end with Correctness judge | ✅ |

**Module 2 done.** You now have three ways to build an eval dataset — hand-curated, trace-derived, and synthetic — and you know when to reach for each.


---
## 📝 Summary

In this notebook, we covered:

### 1. What `generate_evals_df` Does
- Reads document chunks and produces diverse Q&A pairs covering main topics + edge cases.
- Output is a Spark DataFrame already in MLflow's eval schema — `inputs`, `expectations`, `expected_retrieved_context`.

### 2. Bias the Generator
- **`agent_description`** anchors tone and audience.
- **`question_guidelines`** is where you push for edge cases, paraphrase variety, and difficulty mix.

### 3. Cost vs Coverage
- Use **`num_evals=20`** while iterating, **50–100** for production use.
- Persist the generated set to UC so you don't pay for regeneration on every notebook run.

### Next Steps
- **Module 3 — LLM-as-Judge** — turn this dataset into a continuous quality signal.
